# Part 3 — Network-Core Capacity Expansion MILP

### A deterministic circular-economy supply chain, written as a network LP with an integer capacity layer

Parts 1 and 2 built the model as a set of aggregate stage balances. **This notebook rebuilds
the same economics as an explicit network** — nodes, arcs, arc costs — with integer variables
doing one job only: deciding **which nodes exist and how large they are**. Once capacity is
fixed, what remains is a pure minimum-cost flow LP.

That separation is the whole design:

$$\underbrace{\text{binary build} + \text{continuous size}}_{\text{integer layer: sets capacity}}
\;\longrightarrow\;
\underbrace{\text{flows on arcs within those capacities}}_{\text{network LP: cheap to solve}}$$

### What changed from Parts 1 and 2

| | Parts 1 & 2 | **Part 3** |
|---|---|---|
| Capacity | integer count × fixed unit size | **binary build + continuous size** (semi-continuous) |
| Structure | aggregate stage balances | **explicit nodes and arcs** with per-arc cost |
| Periods | annual, or lightly staggered | **variable length**: 8×1yr, 4×3yr, 2×5yr, 1×9yr |
| Horizon | 20 years, reported in full | 39 years, **years 31–39 are a cool-down buffer** |
| Efficiency | vintage-indexed yield | same — vintage-indexed **yield**, not arc cost |
| Learning | SOS2 on cumulative cost | same, plus an explicit **re-mesh** diagnostic |

### Why semi-continuous sizing matters

In Part 2 the lumpy "3 units of 100" decision space made the stochastic and deterministic
strategies land on the same integer, driving the Value of the Stochastic Solution to exactly
zero. Here each site-period gets **one binary** plus a **continuous size** in
$[\text{CAP}_{\min}, \text{CAP}_{\max}]$ — a finer decision space with *fewer* integer
variables. Strictly better on both axes.

### Style note

This notebook is written to be read. Model construction is **inline**, not wrapped in
functions, and uses Gurobi's native looping (`addVars`, `addConstrs` with generator
expressions, `tupledict.sum()` wildcards) so each constraint block reads close to its
algebraic statement. The only function is at the very end, where we deliberately re-solve
variants.

## Formulation reference

Read this once; every code cell below maps onto a line here.

### Sets

| Set | Symbol | Members |
|---|---|---|
| Regions | $r \in \mathcal{R}$ | R1, R2 |
| Stages | $s \in \mathcal{S}$ | MINE → PROC → MFG |
| Nodes | $(s,r) \in \mathcal{N}$ | 6 facilities: one per stage per region |
| Arcs | $(s,r_1,r_2) \in \mathcal{A}$ | from node $(s,r_1)$ to the next stage in $r_2$ (12 arcs) |
| Periods | $p \in \mathcal{P}$ | 15 periods of unequal length spanning 39 years |
| Vintages | $v \in \mathcal{V}$ | $-1$ = inherited legacy asset; $0\ldots14$ = build decided in period $v$ |
| Active triples | $(s,r,v,p) \in \mathcal{X}$ | vintage $v$ at node $(s,r)$ is operating in period $p$ |

### Parameters

| Parameter | Symbol | Meaning |
|---|---|---|
| Period length | $L_p$ | years in period $p$ |
| Period weight | $\omega_p = \sum_{t \in p}(1+\rho)^{-t}$ | **sum** of annual discount factors in the period |
| Discount rate | $\rho$ | 0.05 |
| Asset life | $\Lambda$ | 25 years |
| Lead time | $\ell_s$ | years from decision to operation |
| Capital recovery factor | $\text{CRF} = \frac{\rho(1+\rho)^\Lambda}{(1+\rho)^\Lambda-1}$ | annuitises a lump sum |
| Capex PV multiplier | $\mu_{s,v}$ | $\text{CRF} \times$ sum of discount factors over the asset's operating years inside the horizon |
| Fixed build cost | $F_s$ | per facility, independent of size — never learns |
| Capacity cost | $U_s$ | per unit of capacity |
| Operating cost | $o_s$ | per unit of throughput |
| Transport cost | $\tau_{r_1 r_2}$ | 0.5 intra-region, 2.0 cross-region |
| Yield | $\eta_{s,v,p}$ | output per unit input — **a constraint-matrix coefficient** |
| Demand | $D_{r,p}$ | average annual demand rate in region $r$, period $p$ |
| Shortfall penalty | $\pi$ | cost per unit of unmet demand |
| Size bounds | $\underline{c}, \overline{c}$ | min and max facility size |

### Decision variables

| Variable | Symbol | Domain | Meaning |
|---|---|---|---|
| Build | $y_{s,r,v}$ | $\{0,1\}$ | build a facility at node $(s,r)$, decided in period $v$ |
| Size | $c_{s,r,v}$ | $\ge 0$ | its capacity — **continuous**, but 0 unless $y=1$ |
| Throughput | $x_{s,r,v,p}$ | $\ge 0$ | material processed by vintage $v$ at node $(s,r)$ in period $p$ |
| Flow | $f_{s,r_1,r_2,p}$ | $\ge 0$ | flow on arc $(s,r_1,r_2)$ in period $p$ |
| Shortfall | $u_{r,p}$ | $\ge 0$ | unmet demand |
| Cumulative capacity | $Q_p$ | $\ge Q_0$ | learning-relevant capacity installed by period $p$ |
| Cumulative capex | $C_p$ | $\ge 0$ | cumulative learning-curve cost at $Q_p$ |
| SOS2 weights | $\lambda_{p,k}$ | $[0,1]$ | interpolation on the learning curve |

### Constraints

**Semi-continuous sizing** — capacity is zero, or between the bounds:
$$c_{s,r,v} \le \overline{c}\, y_{s,r,v}, \qquad c_{s,r,v} \ge \underline{c}\, y_{s,r,v}$$

**Capacity limits throughput** — this is the link from the integer layer into the LP:
$$x_{s,r,v,p} \le c_{s,r,v} \quad \forall (s,r,v,p) \in \mathcal{X}$$

**Node output** — yield converts input to output, then it leaves on arcs:
$$\sum_{v} \eta_{s,v,p}\, x_{s,r,v,p} = \sum_{r_2} f_{s,r,r_2,p}$$

**Node input** — what arrives must be processed (for PROC and MFG):
$$\sum_{r_1} f_{s^-,r_1,r,p} = \sum_{v} x_{s,r,v,p}$$

**Demand service:**
$$\sum_{r_1} f_{\text{MFG},r_1,r,p} + u_{r,p} \ge D_{r,p}$$

**Learning curve** (SOS2, on **cumulative** cost):
$$Q_p = \sum_k Q^{bp}_k \lambda_{p,k}, \quad C_p = \sum_k C^{bp}_k \lambda_{p,k},
\quad \sum_k \lambda_{p,k} = 1, \quad \lambda_{p,\cdot} \in \text{SOS2}$$
$$Q_p = Q_0 + \sum_{s \in \text{LEARN}} \sum_{r} \sum_{v \le p} c_{s,r,v}$$

### Objective

$$\min \;\; \underbrace{\sum \mu_{s,v}\big(F_s y_{s,r,v} + U_s c_{s,r,v}\big)}_{\text{capex, annuitised}}
\;+\; \underbrace{\sum_p \mu^{tech}_p (C_p - C_{p-1})}_{\text{learning-curve capex}}
\;+\; \underbrace{\sum \omega_p\, o_s x_{s,r,v,p}}_{\text{operating}}
\;+\; \underbrace{\sum \omega_p\, \tau_{r_1r_2} f_{s,r_1,r_2,p}}_{\text{transport}}
\;+\; \underbrace{\sum \omega_p\, \pi\, u_{r,p}}_{\text{shortfall}}$$

Note $\omega_p$, not 1, on every operating term. **This is the single most common bug in
variable-period models** — a 9-year period weighted as one year understates its operating cost
by roughly 9×.

## 1. Setup

The `pip` build of Gurobi carries a restricted licence capped near 2,000 variables and 2,000
constraints. This model lands at roughly 1,200 variables. To scale up, substitute a WLS
licence:

```python
env = gp.Env(params={"WLSACCESSID": "<your WLS access id>", "WLSSECRET": "<your WLS secret>", "LICENSEID": <your licence id>})
m = gp.Model("part3", env=env)
```

> **Do not paste a licence key into a notebook.** A key committed to a repository is exposed the moment the repository is shared, and deleting it in a later commit does not remove it from history — the only fix is to rotate the key. Read it from an environment variable or a Colab secret instead; `Part4c_exact_MIQP.ipynb` cell 2 shows the pattern.


In [ ]:
!pip install gurobipy --quiet
import math
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})
print("gurobipy", gp.gurobi.version())

## 2. Sets: the network

Three stages in series, two regions, so **six nodes**. An arc is identified by
`(stage, from_region, to_region)` — it leaves the node at that stage in `from_region` and
arrives at the *next* stage in `to_region`. MFG arcs deliver to final demand.

That gives $3 \times 2 \times 2 = 12$ arcs. Intra-region arcs are cheap; cross-region arcs
cost 4× more, which is what makes facility *siting* (not just sizing) an interesting decision.

In [ ]:
REGIONS = ['R1', 'R2']
STAGES  = ['MINE', 'PROC', 'MFG']          # in series: MINE -> PROC -> MFG -> demand
NODES   = [(s, r) for s in STAGES for r in REGIONS]
ARCS    = [(s, r1, r2) for s in STAGES for r1 in REGIONS for r2 in REGIONS]

TRANSPORT = {(r1, r2): (0.5 if r1 == r2 else 2.0)
             for r1 in REGIONS for r2 in REGIONS}

print(f"{len(NODES)} nodes:", NODES)
print(f"{len(ARCS)} arcs, e.g.", ARCS[:3], "...")

## 3. Time: variable-length periods

Investment granularity is fine where decisions bind and coarse where they don't:

| Block | Periods | Years |
|---|---|---|
| 8 × 1-year | 8 | 1–8 |
| 4 × 3-year | 4 | 9–20 |
| 2 × 5-year | 2 | 21–30 |
| 1 × 9-year | 1 | 31–39 |

39 years in **15 periods**. Years 31–39 exist as a **cool-down buffer** — they stop the model
from making artificially myopic choices near the boundary, but we do not report them as
results.

The key object is $\omega_p$, the **sum** of annual discount factors inside period $p$. Every
flow, throughput and demand quantity below is an *annual rate*; $\omega_p$ converts a rate
into that period's discounted contribution. Using a midpoint factor or a bare 1 here is the
classic variable-period error.

In [ ]:
BLOCKS = [(8, 1), (4, 3), (2, 5), (1, 9)]   # (how many periods, years in each)
DR     = 0.05                                # discount rate

LEN, START = [], []
_year = 1
for count, length in BLOCKS:
    for _ in range(count):
        LEN.append(length)
        START.append(_year)
        _year += length

P       = list(range(len(LEN)))
HORIZON = _year - 1
YEARS   = {p: list(range(START[p], START[p] + LEN[p])) for p in P}

# omega_p : SUM of annual discount factors within period p
OMEGA = {p: sum(1 / (1 + DR) ** t for t in YEARS[p]) for p in P}

REPORT_UNTIL = 30          # years 31-39 are the cool-down buffer

print(f"{len(P)} periods spanning {HORIZON} years;  sum of weights = {sum(OMEGA.values()):.3f}")

In [ ]:
pd.DataFrame([dict(period=p, years=f"{START[p]}-{START[p]+LEN[p]-1}", length=LEN[p],
                   omega=round(OMEGA[p], 4),
                   pct_of_objective=round(100*OMEGA[p]/sum(OMEGA.values()), 2))
              for p in P])

## 4. Technology and cost parameters

Two things to notice.

**Capital cost splits in two.** A fixed cost $F_s$ per facility (permitting, site works,
grid/road connection) that does **not** depend on size and never falls with learning, plus a
capacity cost $U_s$ per unit. This is what makes semi-continuous sizing economically
meaningful — without the fixed part, the model would build many tiny facilities.

**Capex is annuitised, not charged as a lump sum.** $\mu_{s,v}$ charges
$\text{CRF} \times \text{cost}$ in each operating year that falls inside the horizon. Because
an annuity discounts back to exactly its principal over a full life, the fraction of *cost*
charged inside the horizon equals the fraction of *value* captured — so the end-of-horizon
truncation bias cancels instead of suppressing late builds.

In [ ]:
LIFE    = 25                                     # asset life, years
LEAD    = {'MINE': 1, 'PROC': 2, 'MFG': 2}       # decision -> operation, years
CAP_MIN, CAP_MAX = 60.0, 260.0                   # facility size bounds

FIXED   = {'MINE':  900.0, 'PROC': 1500.0, 'MFG': 1300.0}   # per facility
UNIT    = {'MINE':    7.0, 'PROC':   11.0, 'MFG':    9.5}   # per unit of capacity
OPERATE = {'MINE':    1.2, 'PROC':    2.0, 'MFG':    2.4}   # per unit of throughput

CRF = DR * (1 + DR) ** LIFE / ((1 + DR) ** LIFE - 1)

ONLINE = {(s, p): START[p] + LEAD[s] for s in STAGES for p in P}

# mu[s,v] : PV of $1 of capital, charged as CRF per operating year inside the horizon
MU = {(s, v): CRF * sum(1 / (1 + DR) ** t
                        for t in range(ONLINE[s, v], ONLINE[s, v] + LIFE)
                        if t <= HORIZON)
      for s in STAGES for v in P}

print(f"CRF ({LIFE} yr, {DR:.0%}) = {CRF:.5f}")
print("MU for PROC by decision period:",
      {v: round(MU['PROC', v], 3) for v in [0, 4, 8, 11, 14]})

## 5. Legacy assets and demand

The network starts **brownfield**. Six inherited facilities retire on a staggered schedule
(years 9, 12, 14, 16, 19, 24) rather than all at once — that spreads replacement pressure
across the horizon instead of creating a single cliff, which would make every result an
artefact of the boundary.

Demand is an **annual rate**, averaged over each period's years. Region 2 grows faster than
Region 1 (2.6% vs 0.8%), so over the horizon the network's centre of gravity should shift —
which is exactly what the cross-region transport premium makes costly, and therefore
interesting.

Unmet demand is allowed at penalty $\pi$. That is not a numerical convenience: it keeps the
model feasible when lead times make a shortfall physically unavoidable, and it turns
"infeasible" into a **reportable quantity**.

In [ ]:
LEGACY_CAP = {('MINE','R1'):200, ('MINE','R2'):180,
              ('PROC','R1'):170, ('PROC','R2'):150,
              ('MFG', 'R1'):130, ('MFG', 'R2'):120}
LEGACY_RETIRE_YEAR = {('MINE','R1'): 9, ('MINE','R2'):14,
                      ('PROC','R1'):12, ('PROC','R2'):19,
                      ('MFG', 'R1'):16, ('MFG', 'R2'):24}
LEGACY_BUILD_YEAR = -8                  # inherited assets are already 8 years old

DEMAND = {}
for r, base, growth in [('R1', 100.0, 0.008), ('R2', 70.0, 0.026)]:
    for p in P:
        DEMAND[r, p] = sum(base * (1 + growth) ** (t - 1) for t in YEARS[p]) / LEN[p]

SHORTFALL_PENALTY = 90.0

print("demand rate  yr 1: ", {r: round(DEMAND[r, 0], 1) for r in REGIONS})
print("demand rate  yr 31+:", {r: round(DEMAND[r, P[-1]], 1) for r in REGIONS})

## 6. Efficiency: two channels, both vintage-indexed

This is the part worth reading carefully, because it is where the formulation makes a
substantive modelling claim.

**Yield, not cost.** $\eta$ is *output per unit of input* — it belongs in the **constraint
matrix**. A better recovery rate means you need less ore for the same finished output, which
propagates upstream and changes how much mining capacity you must build. Representing
efficiency as a cheaper arc cost instead would only be valid if upstream capacity never binds
— and in a capacity-expansion model, upstream capacity binding is precisely the case of
interest.

**Two independent improvement channels:**

$$\eta^{new}(v) = \bar\eta - (\bar\eta - \eta_0)(1-\alpha)^{\,\text{year}(v)-1}
\qquad \text{(frontier: newer builds are better)}$$

$$\eta(v,p) = \min\Big\{\eta^{new}(v) + \bar\Delta,\;\;
\bar\eta - \big(\bar\eta - \eta^{new}(v)\big)(1-\beta)^{\,\text{age}}\Big\}
\qquad \text{(learning by operating)}, \quad \beta < \alpha$$

Two properties fall out of this form, and both matter:

- Because $\beta < \alpha$ and each asset closes the *remaining* gap to the same ceiling
  $\bar\eta$, an older facility improves over its life but **never overtakes a newer vintage**.
- Nothing crosses $\bar\eta$, and $\bar\Delta$ caps how much any single asset can gain from
  retrofits.

Critically, $\eta$ is a **parameter**, not a variable — a dictionary lookup, zero additional
columns in the model. Binning assets into "new / mid / old" node groups and unlocking them
with big-M constraints would cost extra binaries, extra constraints, a weaker LP relaxation,
*and* it would only approximate what this exact lookup gives for free.

In [ ]:
ETA_CEIL  = {'MINE':0.92,  'PROC':0.95,  'MFG':0.93}    # eta_bar
ETA_BASE  = {'MINE':0.86,  'PROC':0.80,  'MFG':0.78}    # eta_0, a year-1 build
ALPHA     = {'MINE':0.000, 'PROC':0.030, 'MFG':0.025}   # frontier gain per year of vintage
BETA      = {'MINE':0.000, 'PROC':0.010, 'MFG':0.008}   # within-life gain per year of age
DELTA_BAR = {'MINE':0.02,  'PROC':0.05,  'MFG':0.05}    # cap on lifetime retrofit gain
ETA_FLOOR = 0.60

VINTAGES   = [-1] + P                       # -1 is the inherited legacy cohort
BUILD_YEAR = {v: (LEGACY_BUILD_YEAR if v == -1 else START[v]) for v in VINTAGES}

ETA = {}
for s in STAGES:
    for v in VINTAGES:
        frontier = ETA_CEIL[s] - (ETA_CEIL[s] - ETA_BASE[s]) * (1 - ALPHA[s]) ** (BUILD_YEAR[v] - 1)
        frontier = max(ETA_FLOOR, min(frontier, ETA_CEIL[s]))
        for p in P:
            age  = max(0, START[p] - BUILD_YEAR[v])
            aged = ETA_CEIL[s] - (ETA_CEIL[s] - frontier) * (1 - BETA[s]) ** age
            ETA[s, v, p] = max(ETA_FLOOR, min(frontier + DELTA_BAR[s], aged))

print("PROC yield, by vintage and operating period:")
print(pd.DataFrame({f'vintage {v}': [round(ETA['PROC', v, p], 4) for p in P]
                    for v in [-1, 0, 6, 11]},
                   index=[f'yr {START[p]}' for p in P]).T.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.4))
for j, v in enumerate([-1, 0, 4, 8, 11, 14]):
    yrs = [START[p] for p in P if START[p] >= max(1, BUILD_YEAR[v])]
    ax.plot(yrs, [ETA['PROC', v, p] for p in P if START[p] >= max(1, BUILD_YEAR[v])],
            'o-', lw=2.3, ms=5, color=plt.cm.viridis(j/5),
            label='legacy' if v == -1 else f'vintage yr {START[v]}')
ax.axhline(ETA_CEIL['PROC'], ls='--', lw=2, color='k')
ax.text(2, ETA_CEIL['PROC'] + 0.004, r'ceiling $\bar\eta$', fontsize=11)
ax.set_xlabel('operating year'); ax.set_ylabel(r'processing yield $\eta(v,p)$')
ax.set_title('Curves rise with age but never cross: newer vintages stay ahead')
ax.legend(fontsize=10, ncol=2); plt.tight_layout(); plt.show()

## 7. Which vintages are operating when?

Before writing constraints we enumerate the **active triples** $\mathcal{X}$: combinations of
node, vintage and period where that vintage is actually operating. A vintage decided in
period $v$ comes online at $\text{START}_v + \ell_s$ and runs for $\Lambda$ years; the legacy
cohort runs until its retirement year.

Building this as a list comprehension keeps the model sparse — we never create a throughput
variable for an asset that does not exist yet or has already retired. `BUILD` is filtered the
same way: a decision whose asset could never come online inside the horizon is excluded
outright rather than left for the solver to zero out.

In [ ]:
ACTIVE = [(s, r, v, p)
          for (s, r) in NODES for v in VINTAGES for p in P
          if (v == -1 and START[p] <= LEGACY_RETIRE_YEAR[s, r])
          or (v >= 0 and ONLINE[s, v] <= START[p] <= ONLINE[s, v] + LIFE - 1)]

# vintages operating at each node in each period -- used inside the balance constraints
VIN = {(s, r, p): [v for (ss, rr, v, pp) in ACTIVE if (ss, rr, pp) == (s, r, p)]
       for (s, r) in NODES for p in P}

# a build decision is only meaningful if the asset comes online inside the horizon
BUILD = [(s, r, v) for (s, r) in NODES for v in P if ONLINE[s, v] <= HORIZON]

print(f"{len(ACTIVE)} active (node, vintage, period) triples")
print(f"{len(BUILD)} candidate build decisions  ->  {len(BUILD)} binaries")

## 8. The learning curve

Capex for processing and manufacturing falls with **cumulative installed capacity** (Wright's
law), floored so it cannot decline forever:

$$U(Q) = \max\Big\{\phi U_0,\; U_0 (Q/Q_0)^{-b}\Big\}, \qquad b = -\log_2(1 - LR)$$

Two implementation points that are easy to get wrong.

**Linearise the *cumulative* cost, not the unit cost.** We need
$C(Q) = \int_{Q_0}^{Q} U(q)\,dq$, then charge the period-over-period increment
$C_p - C_{p-1}$.

**SOS2 is not optional.** $C(Q)$ is concave and increasing, and we are minimising, so an LP
relaxation with free $\lambda$ would mix non-adjacent breakpoints and ride the chord *below*
the true curve — handing the model a cost reduction it never earned. SOS2 restricts $\lambda$
to at most two **adjacent** nonzeros, which is what makes the approximation valid.

The mesh spans $[Q_0, Q_0 + 1000]$. We chose that upper bound *after* an initial solve
revealed where cumulative capacity actually lands — §13 shows the diagnostic. Setting it from
a theoretical worst case would put the coarsest chords exactly where the solution lives.

In [ ]:
LEARN_STAGES  = ['PROC', 'MFG']
LEARNING_RATE = 0.20          # cost fall per doubling of cumulative capacity
Q_START       = 400.0         # incumbent cumulative capacity
Q_ADD         = 1000.0        # mesh headroom above Q_START
COST_FLOOR    = 0.55          # floor as a fraction of the starting unit cost
N_BREAK       = 9

_b  = -math.log2(1 - LEARNING_RATE)
_U0 = sum(UNIT[s] for s in LEARN_STAGES) / len(LEARN_STAGES)

# the only helper functions in the notebook: the curve and its integral
def unit_cost(q):
    return max(COST_FLOOR * _U0, _U0 * (q / Q_START) ** (-_b))

def cumulative_cost(q, n=600):
    if q <= Q_START:
        return 0.0
    h = (q - Q_START) / n
    return sum(0.5 * (unit_cost(Q_START + i*h) + unit_cost(Q_START + (i+1)*h)) * h
               for i in range(n))

K   = list(range(N_BREAK))
QBP = [Q_START + Q_ADD * k / (N_BREAK - 1) for k in K]
CBP = [cumulative_cost(q) for q in QBP]

# PROC and MFG share LEAD = 2, so one PV multiplier is exact for both
MU_TECH = {p: MU['PROC', p] for p in P}

print("Q breakpoints:", [round(q) for q in QBP])
print("unit cost    :", [round(unit_cost(q), 2) for q in QBP])

## 9. Variables

Five families, declared with `addVars` so each returns a `tupledict` — which gives us
`.sum()` with wildcards later, and keeps the constraint blocks readable.

| Code | Symbol | Notes |
|---|---|---|
| `build[s,r,v]` | $y_{s,r,v}$ | binary — **the only integer variables in the model** |
| `size[s,r,v]` | $c_{s,r,v}$ | continuous capacity, forced to 0 unless `build` is 1 |
| `thr[s,r,v,p]` | $x_{s,r,v,p}$ | throughput, indexed by **vintage** so yield can differ by cohort |
| `flow[s,r1,r2,p]` | $f_{s,r_1,r_2,p}$ | arc flow |
| `short[r,p]` | $u_{r,p}$ | unmet demand |

Note the variable counts: 90 binaries against roughly 1,100 continuous variables. That ratio
is the point of the design — branch-and-bound only has to search the small integer layer, and
everything else resolves as an LP.

In [ ]:
m = gp.Model('part3_network')
m.Params.OutputFlag = 0
m.Params.MIPGap = 0.005

build = m.addVars(BUILD,   vtype=GRB.BINARY,        name='build')
size  = m.addVars(BUILD,   lb=0.0, ub=CAP_MAX,     name='size')
thr   = m.addVars(ACTIVE,  lb=0.0,                 name='thr')
flow  = m.addVars(ARCS, P, lb=0.0,                 name='flow')
short = m.addVars(REGIONS, P, lb=0.0,              name='short')

m.update()
print(f"{m.NumVars} variables, of which {m.NumBinVars} binary")

## 10. Semi-continuous sizing

$$c_{s,r,v} \le \overline{c}\, y_{s,r,v} \qquad\qquad c_{s,r,v} \ge \underline{c}\, y_{s,r,v}$$

Together these say: **either build nothing, or build something between $\underline{c}$ and
$\overline{c}$.** The lower bound is what prevents the model from evading the fixed cost $F_s$
by building an arbitrarily small facility.

Note the big-M here is $\overline{c}$ — the tightest valid bound, not a generic large number.
A loose big-M is the most common cause of hopeless branch-and-bound performance in facility
location problems, because it destroys the LP relaxation.

In [ ]:
m.addConstrs((size[s, r, v] <= CAP_MAX * build[s, r, v] for (s, r, v) in BUILD),
             name='size_upper')

m.addConstrs((size[s, r, v] >= CAP_MIN * build[s, r, v] for (s, r, v) in BUILD),
             name='size_lower')
print("semi-continuous sizing constraints added")

## 11. Capacity limits throughput — the integer/LP interface

$$x_{s,r,v,p} \le c_{s,r,v} \qquad \forall (s,r,v,p) \in \mathcal{X}$$

This single constraint family is the hinge of the whole formulation. It is the *only* place
the integer layer touches the flow problem. Fix `size`, and every constraint from here down is
linear in continuous variables — a minimum-cost flow problem.

For the legacy cohort ($v = -1$) the right-hand side is a **parameter**, not a variable: the
inherited capacity is given, not chosen.

In [ ]:
m.addConstrs((thr[s, r, v, p] <= (LEGACY_CAP[s, r] if v == -1 else size[s, r, v])
              for (s, r, v, p) in ACTIVE), name='capacity')
print("capacity constraints added")

## 12. Network flow balance

Three families, one per structural role.

**Node output.** Throughput is converted at the vintage's own yield, and the result leaves on
outbound arcs:
$$\sum_{v} \eta_{s,v,p}\, x_{s,r,v,p} \;=\; \sum_{r_2} f_{s,r,r_2,p}$$

This is where $\eta$ does its work as a constraint coefficient. `flow.sum(s, r, '*', p)` is
Gurobi's wildcard sum over the third index — it reads almost exactly like the algebra.

**Node input.** For PROC and MFG, everything arriving must be processed by some vintage:
$$\sum_{r_1} f_{s^-,r_1,r,p} \;=\; \sum_{v} x_{s,r,v,p}$$

MINE has no inbound arc — ore comes out of the ground, bounded only by capacity.

**Demand service.** Manufacturing output plus shortfall must cover demand:
$$\sum_{r_1} f_{\text{MFG},r_1,r,p} + u_{r,p} \;\ge\; D_{r,p}$$

In [ ]:
# node output: yield-converted throughput leaves on outbound arcs
m.addConstrs((gp.quicksum(ETA[s, v, p] * thr[s, r, v, p] for v in VIN[s, r, p])
              == flow.sum(s, r, '*', p)
              for (s, r) in NODES for p in P), name='node_output')

# node input: what arrives at PROC / MFG must be processed
m.addConstrs((flow.sum('MINE', '*', r, p)
              == gp.quicksum(thr['PROC', r, v, p] for v in VIN['PROC', r, p])
              for r in REGIONS for p in P), name='input_proc')

m.addConstrs((flow.sum('PROC', '*', r, p)
              == gp.quicksum(thr['MFG', r, v, p] for v in VIN['MFG', r, p])
              for r in REGIONS for p in P), name='input_mfg')

# demand service, with shortfall allowed at a penalty
m.addConstrs((flow.sum('MFG', '*', r, p) + short[r, p] >= DEMAND[r, p]
              for r in REGIONS for p in P), name='demand')
print("network balance constraints added")

## 13. Endogenous learning via SOS2

$Q_p$ accumulates learning-relevant capacity; $C_p$ reads the cumulative cost off the
piecewise curve; $\lambda_{p,\cdot}$ interpolates under an SOS2 restriction.

`addSOS` is the one place a Python loop is unavoidable — it takes a single variable list per
call, so there is no vectorised form.

Because $Q_p$ sums `size` over **all vintages $v \le p$**, learning is genuinely endogenous:
building earlier lowers the cost of building later, and the model can choose to buy the
reduction down. That is the difference from a calendar-indexed cost decline, which would hand
the model cheaper technology whether or not it ever built anything.

In [ ]:
Q   = m.addVars(P, lb=Q_START, ub=Q_START + Q_ADD, name='Qcum')
Ccm = m.addVars(P, lb=0.0,                          name='Ccum')
lam = m.addVars(P, K, lb=0.0, ub=1.0,               name='lam')

m.addConstrs((lam.sum(p, '*') == 1 for p in P), name='sos_convexity')

m.addConstrs((Q[p] == gp.quicksum(QBP[k] * lam[p, k] for k in K) for p in P),
             name='sos_Q')

m.addConstrs((Ccm[p] == gp.quicksum(CBP[k] * lam[p, k] for k in K) for p in P),
             name='sos_C')

m.addConstrs((Q[p] == Q_START + gp.quicksum(size[s, r, v] for (s, r, v) in BUILD
                                            if s in LEARN_STAGES and v <= p)
              for p in P), name='cumulative_capacity')

for p in P:                                   # SOS2 must be added one set at a time
    m.addSOS(GRB.SOS_TYPE2, [lam[p, k] for k in K])

print("learning curve added:", N_BREAK, "breakpoints x", len(P), "periods")

## 14. Objective

$$\min \;\; \underbrace{\sum \mu_{s,v}\big(F_s y + U_s c\big)}_{\text{capex}}
+ \underbrace{\sum_p \mu^{tech}_p (C_p - C_{p-1})}_{\text{learning capex}}
+ \underbrace{\sum \omega_p o_s x}_{\text{operating}}
+ \underbrace{\sum \omega_p \tau f}_{\text{transport}}
+ \underbrace{\sum \omega_p \pi u}_{\text{shortfall}}$$

Three details worth stating explicitly:

- **$\mu$ on capital, $\omega$ on flow.** Capital is annuitised over the asset's operating
  years; operating quantities are rates and get the period's summed discount weight. Mixing
  these up is the variable-period bug.
- **The fixed cost never learns.** $F_s$ is charged at full price in every period. Only the
  capacity-proportional part of PROC and MFG capex flows through the learning curve, and
  mining — a mature technology — is excluded from learning entirely.
- **Capital cost is locked at the vintage.** An asset built in period 3 keeps paying period-3
  prices for its whole life. Letting the charge float down as later learning accrues would
  retroactively cheapen sunk assets.

In [ ]:
capex = gp.quicksum(MU[s, v] * FIXED[s] * build[s, r, v] for (s, r, v) in BUILD) \
      + gp.quicksum(MU[s, v] * UNIT[s] * size[s, r, v]
                    for (s, r, v) in BUILD if s not in LEARN_STAGES)

learn = gp.quicksum(MU_TECH[p] * (Ccm[p] - (Ccm[p-1] if p > 0 else 0.0)) for p in P)

operate = gp.quicksum(OMEGA[p] * OPERATE[s] * thr[s, r, v, p]
                      for (s, r, v, p) in ACTIVE)

transport = gp.quicksum(OMEGA[p] * TRANSPORT[r1, r2] * flow[s, r1, r2, p]
                        for (s, r1, r2) in ARCS for p in P)

penalty = gp.quicksum(OMEGA[p] * SHORTFALL_PENALTY * short[r, p]
                      for r in REGIONS for p in P)

m.setObjective(capex + learn + operate + transport + penalty, GRB.MINIMIZE)
m.update()
print(f"{m.NumVars} variables | {m.NumConstrs} constraints | {m.NumBinVars} binaries")

## 15. Solve

In [ ]:
m.optimize()
print("status", m.Status, " objective", round(m.ObjVal, 1),
      " MIP gap", f"{m.MIPGap:.4%}", " nodes", int(m.NodeCount))
print()
for label, expr in [('capex (fixed + mining)', capex), ('capex (learning curve)', learn),
                    ('operating', operate), ('transport', transport),
                    ('shortfall penalty', penalty)]:
    print(f"  {label:26s} {expr.getValue():10.1f}")
print(f"\n  total unmet demand {sum(short[r, p].X for r in REGIONS for p in P):.2f} units")

In [ ]:
plan = pd.DataFrame([dict(stage=s, region=r, period=v, year=START[v],
                          size=round(size[s, r, v].X, 1),
                          in_report_window=START[v] <= REPORT_UNTIL)
                     for (s, r, v) in BUILD if build[s, r, v].X > 0.5]
                    ).sort_values(['year', 'stage'])
plan

Note the `size` column: the facilities are **not** all the same size, and none of them is a
round multiple of anything. That is semi-continuous sizing doing its job — the model picks the
capacity it wants, subject only to the fixed cost making very small builds uneconomic.

Also note `in_report_window`. Decisions in years 31–39 belong to the cool-down buffer and
should not be quoted as findings.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
colours = {'MINE':'#8e44ad', 'PROC':'#2471a3', 'MFG':'#196f3d'}
for s in STAGES:
    series = []
    for p in P:
        total = sum((LEGACY_CAP[ss, rr] if v == -1 else size[ss, rr, v].X)
                    for (ss, rr, v, pp) in ACTIVE if pp == p and ss == s)
        series.append(total)
    ax[0].plot([START[p] for p in P], series, 'o-', lw=2.5, ms=6,
               color=colours[s], label=s)
for (s, r) in NODES:
    ax[0].axvline(LEGACY_RETIRE_YEAR[s, r], ls=':', color='grey', alpha=0.45)
ax[0].axvspan(REPORT_UNTIL, HORIZON, color='grey', alpha=0.12)
ax[0].text(REPORT_UNTIL + 0.6, ax[0].get_ylim()[1]*0.35, 'cool-down\nbuffer',
           fontsize=10, color='dimgrey')
ax[0].set_xlabel('year'); ax[0].set_ylabel('installed capacity')
ax[0].set_title('Capacity by stage (dotted = legacy retirements)'); ax[0].legend(fontsize=10)

served = [sum(flow.sum('MFG', '*', r, p).getValue() for r in REGIONS) for p in P]
demand = [sum(DEMAND[r, p] for r in REGIONS) for p in P]
ax[1].plot([START[p] for p in P], demand, 's--', lw=2.5, color='#c0392b', label='demand')
ax[1].plot([START[p] for p in P], served, 'o-', lw=2.5, color='#196f3d', label='served')
ax[1].axvspan(REPORT_UNTIL, HORIZON, color='grey', alpha=0.12)
ax[1].set_xlabel('year'); ax[1].set_ylabel('annual rate'); ax[1].legend(fontsize=10)
ax[1].set_title('Demand vs delivery')
plt.tight_layout(); plt.show()

## 16. Diagnostic: is the SOS2 mesh actually binding?

A piecewise approximation can be quietly useless. If every $\lambda_{p,\cdot}$ lands on a
single breakpoint you have built a step function, not a curve — and if non-adjacent
breakpoints carry weight, SOS2 is not being enforced at all.

Check both, and also check **where the breakpoints sit relative to where the solution goes.**

In [ ]:
rows = []
for p in P:
    nz = {k: round(lam[p, k].X, 3) for k in K if lam[p, k].X > 1e-6}
    ks = sorted(nz)
    status = ('interpolating' if len(ks) == 2 and ks[1] - ks[0] == 1
              else 'at a breakpoint' if len(ks) == 1 else 'ADJACENCY VIOLATION')
    rows.append(dict(period=p, year=START[p], Q=round(Q[p].X, 1),
                     nonzero_lambda=str(nz), status=status))
used   = {k for p in P for k in K if lam[p, k].X > 1e-6}
print(pd.DataFrame(rows).to_string(index=False))
print(f"\nmax Q reached {max(Q[p].X for p in P):.1f}  (mesh upper bound {Q_START + Q_ADD:.0f})")
print(f"breakpoints never used: {sorted(set(K) - used)} of {len(K)}")

Adjacency should never be violated, and several periods should be interpolating — that
confirms the curve is doing real work rather than sitting on a corner.

**On the unused breakpoints:** the first version of this notebook set `Q_ADD` from a
theoretical worst case (every node, every period, at maximum size), which put the mesh upper
bound near 2,200 while the solution only reached ~1,174. Four of nine breakpoints went unused
and the chords were coarsest exactly where the answer lived — the approximation understated
concave capex, and re-meshing tighter *raised* the objective slightly. That is the direction
to expect: a coarse mesh on a concave cost curve always errs optimistic.

The practical rule: **solve once, read the realised $Q$, then re-mesh around it.** One extra
solve for a mesh concentrated where the solution actually goes.

## 17. Comparisons

Now we re-solve variants, which is the one place a function is genuinely warranted — three
near-identical models differing in two switches. Everything inside is the same code as above.

- **`capex_mode`** — annuitised (CRF per operating year) vs lump-sum at the decision year
- **`learning`** — endogenous SOS2 curve vs a flat capacity cost

In [ ]:
def solve_variant(capex_mode='annualized', learning='endogenous', mipgap=0.005):
    v = gp.Model(); v.Params.OutputFlag = 0; v.Params.MIPGap = mipgap
    b  = v.addVars(BUILD, vtype=GRB.BINARY); c = v.addVars(BUILD, lb=0, ub=CAP_MAX)
    x  = v.addVars(ACTIVE, lb=0); f = v.addVars(ARCS, P, lb=0); u = v.addVars(REGIONS, P, lb=0)

    v.addConstrs(c[s, r, w] <= CAP_MAX * b[s, r, w] for (s, r, w) in BUILD)
    v.addConstrs(c[s, r, w] >= CAP_MIN * b[s, r, w] for (s, r, w) in BUILD)
    v.addConstrs(x[s, r, w, p] <= (LEGACY_CAP[s, r] if w == -1 else c[s, r, w])
                 for (s, r, w, p) in ACTIVE)
    v.addConstrs(gp.quicksum(ETA[s, w, p] * x[s, r, w, p] for w in VIN[s, r, p])
                 == f.sum(s, r, '*', p) for (s, r) in NODES for p in P)
    v.addConstrs(f.sum('MINE', '*', r, p)
                 == gp.quicksum(x['PROC', r, w, p] for w in VIN['PROC', r, p])
                 for r in REGIONS for p in P)
    v.addConstrs(f.sum('PROC', '*', r, p)
                 == gp.quicksum(x['MFG', r, w, p] for w in VIN['MFG', r, p])
                 for r in REGIONS for p in P)
    v.addConstrs(f.sum('MFG', '*', r, p) + u[r, p] >= DEMAND[r, p]
                 for r in REGIONS for p in P)

    mult = ({(s, w): MU[s, w] for (s, r, w) in BUILD} if capex_mode == 'annualized'
            else {(s, w): 1 / (1 + DR) ** START[w] for (s, r, w) in BUILD})
    obj = gp.quicksum(mult[s, w] * FIXED[s] * b[s, r, w] for (s, r, w) in BUILD) \
        + gp.quicksum(mult[s, w] * UNIT[s] * c[s, r, w]
                      for (s, r, w) in BUILD if s not in LEARN_STAGES)

    if learning == 'none':
        obj += gp.quicksum(mult[s, w] * UNIT[s] * c[s, r, w]
                           for (s, r, w) in BUILD if s in LEARN_STAGES)
    else:
        Qv = v.addVars(P, lb=Q_START, ub=Q_START + Q_ADD)
        Cv = v.addVars(P, lb=0); lv = v.addVars(P, K, lb=0, ub=1)
        v.addConstrs(lv.sum(p, '*') == 1 for p in P)
        v.addConstrs(Qv[p] == gp.quicksum(QBP[k] * lv[p, k] for k in K) for p in P)
        v.addConstrs(Cv[p] == gp.quicksum(CBP[k] * lv[p, k] for k in K) for p in P)
        v.addConstrs(Qv[p] == Q_START + gp.quicksum(c[s, r, w] for (s, r, w) in BUILD
                                                    if s in LEARN_STAGES and w <= p)
                     for p in P)
        for p in P:
            v.addSOS(GRB.SOS_TYPE2, [lv[p, k] for k in K])
        obj += gp.quicksum(MU_TECH[p] * (Cv[p] - (Cv[p-1] if p > 0 else 0.0)) for p in P)

    obj += gp.quicksum(OMEGA[p] * OPERATE[s] * x[s, r, w, p] for (s, r, w, p) in ACTIVE)
    obj += gp.quicksum(OMEGA[p] * TRANSPORT[r1, r2] * f[s, r1, r2, p]
                       for (s, r1, r2) in ARCS for p in P)
    obj += gp.quicksum(OMEGA[p] * SHORTFALL_PENALTY * u[r, p] for r in REGIONS for p in P)
    v.setObjective(obj, GRB.MINIMIZE); v.optimize()
    return dict(obj=v.ObjVal,
                builds=sum(1 for k in BUILD if b[k].X > 0.5),
                capacity=round(sum(c[k].X for k in BUILD), 1),
                unmet=round(sum(u[r, p].X for r in REGIONS for p in P), 1),
                first_year=min([START[w] for (s, r, w) in BUILD if b[s, r, w].X > 0.5],
                               default=None))

In [ ]:
rows = []
for mode in ['lumpsum', 'annualized']:
    rows.append(dict(variant=f'capex = {mode}', **solve_variant(capex_mode=mode)))
for lm in ['none', 'endogenous']:
    rows.append(dict(variant=f'learning = {lm}', **solve_variant(learning=lm)))
pd.DataFrame(rows)

### Reading the comparison

**Endogenous learning is cheaper than no learning, and that is legitimate here** — the
reduction has to be bought down through actual installed capacity, unlike a calendar-indexed
decline which would hand it over for free.

**The lump-sum penalty is small — around half a percent — and that is the cool-down buffer
working.** In Part 1 (20-year horizon, 20-year lives, no buffer) lump-sum accounting refused
to build in the final period at all and absorbed an order of magnitude more unmet demand.
Here the horizon runs to 39 years while all the reported decisions sit inside year 30, so a
build in the reporting window still captures most of its asset life inside the model. The
truncation bias has been pushed out into the buffer, where it cannot contaminate the results.

That is worth being precise about: annuitising capex and adding a cool-down buffer are
**two different fixes for the same bias**, and applying both is why the effect nearly
disappears. Remove the buffer — set the horizon to 30 years — and the lump-sum gap reopens.

## 18. How this compares to Parts 1 and 2

| | Parts 1 & 2 | Part 3 |
|---|---|---|
| Binaries | count × fixed unit | **one per node-period**, size continuous |
| Capacity grid | multiples of 80–120 | continuous in $[60, 260]$ |
| Structure | aggregate stage balances | **explicit network**, per-arc cost |
| Siting | implicit | arc costs make cross-region sourcing expensive |
| Periods | annual / lightly staggered | 8×1, 4×3, 2×5, 1×9 |
| Buffer | none in Part 1 | years 31–39 |
| Code style | helper functions | inline `addConstrs`, readable as algebra |

### What this formulation buys

- **A genuine LP core.** With `size` fixed the residual is a minimum-cost flow problem. That
  opens **Benders decomposition** (the L-shaped method): an integer master proposes capacities,
  an LP subproblem prices them and returns a cut. For a *stochastic* extension with integer
  first stage and LP recourse this is usually preferable to progressive hedging, because the
  subproblems are convex with exact duals — where PH on a mixed-integer subproblem is a
  heuristic needing a $\rho$ sweep.
- **Fewer integers, finer decisions.** 90 binaries here versus 270 in Part 1, with a *finer*
  attainable capacity grid. This is what would remove the exact-zero VSS artefact from Part 2.
- **Readable constraints.** `flow.sum('MFG', '*', r, p) + short[r, p] >= DEMAND[r, p]` is
  close to unambiguous.

### Costs and caveats

- **More continuous variables.** Vintage-indexed throughput is ~640 columns. Cheap for an LP,
  but it grows as periods × life.
- **Recognising the LP core does not by itself buy speed.** Branch-and-bound already solves an
  LP at every node with integers fixed. The gain comes from *explicitly* decomposing.
- **`Q_ADD` is a modelling restriction, not just a mesh choice.** It caps cumulative learning
  capacity. Verify it is not binding (§16 does).
- **Fixed-cost realism drives the size distribution.** If $F_s$ is too small the model builds
  many minimum-size facilities; too large and it builds few maximum-size ones. Calibrate it.
- **One facility per node-period.** Two plants of different sizes in the same region and
  period is not expressible. Add a second index if that matters.

### Extensions worth trying

- `REPORT_UNTIL = HORIZON` and re-run the capex comparison — the lump-sum bias should reopen
- `SHORTFALL_PENALTY = 30` — cheap shortfall, and the model will rationally under-build
- `LEARNING_RATE = 0.35` — does build *timing* shift, or only cost?
- Make `TRANSPORT` cross-region cheaper and watch siting consolidate into one region
- Add scrap return arcs from MFG back to PROC for a true circular loop, with a recovery yield
- Wrap this in the Part 2 scenario machinery for a stochastic version — the LP core makes
  Benders/L-shaped the natural algorithm